<a href="https://colab.research.google.com/github/rionkuri22/Re-collect_Scottylabs/blob/main/recollect_jsons.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 4.2 MB/s eta 0:00:00


# Test: Rion

Extract text from PDF using pyPDF2. Raw dump ONLY. No structure

In [5]:
import PyPDF2

def extract_text_from_pdf(pdf_path):
    text = ""
    try:
        with open(pdf_path, 'rb') as file:
            reader = PyPDF2.PdfReader(file)
            for page_num in range(len(reader.pages)):
                page = reader.pages[page_num]
                text += page.extract_text() or ""
    except Exception as e:
        print(f"Error reading PDF: {e}")
        return None
    return text

pdf_file_path = '/content/drive/MyDrive/CMU/class (for Gemini use)/Resume (File responses)/Rion Kurihara resume (ENG).pdf'

resume_text = extract_text_from_pdf(pdf_file_path)

if resume_text:
    print("Text extracted successfully. First 500 characters:\n", resume_text[:500])
else:
    print("Failed to extract text from PDF. ")

Text extracted successfully. First 500 characters:
  Rion Kurihara  rion.kurihara622@gmail.com | 070-1389-0622  Education  Keio University     September 2023 - (September 2027)  B.A. in Policy Management (English degree program); Concentration: AI in Politics and Business  Carnegie Mellon University (academic exchange)    August 2025 - May 2026  Information Systems Major; Concentration: AI/ML + Human-AI-Interaction + Product management  Recognized on the Dean’s List with High Honors for outstanding academic performance  Recipient of the Gyomu Sup


Add extracted resume text to json

In [6]:
import json

if resume_text:
    person_name = "Rion Kurihara"

    # Prepare the data for JSON. This is a basic structure.
    # For a more organized resume, you would parse 'resume_text' into more specific fields.
    resume_data = {
        "person_name": person_name,
        "extracted_text": resume_text,
        "structured_info": "<ADD_PARSED_STRUCTURED_INFORMATION_HERE>"
        # Example: "experience": [{"title": "...", "company": "..."}],
        #          "education": [{"degree": "...", "university": "..."}],
        #          "skills": ["Python", "SQL"]
    }

    # Define the path to save the JSON file in Google Drive
    # The filename will be based on the person's name
    json_filename = f"{person_name.replace(' ', '_').lower()}_resume.json"
    json_file_path = f'/content/drive/MyDrive/{json_filename}'

    # Save the JSON data to the file
    try:
        with open(json_file_path, 'w') as f:
            json.dump(resume_data, f, indent=4)
        print(f"JSON data successfully saved to {json_file_path}")
    except Exception as e:
        print(f"Error saving JSON to Drive: {e}")
else:
    print("Cannot create JSON as no text was extracted from the PDF.")

JSON data successfully saved to /content/drive/MyDrive/rion_kurihara_resume.json


Add Github data to json

In [9]:
import requests
import json
import base64
# For adding delays to be polite to the API
import time

github_username = "rionkuri22"
github_api_url_user = f"https://api.github.com/users/{github_username}"
github_api_url_repos = f"https://api.github.com/users/{github_username}/repos"

print(f"Fetching GitHub data for user: {github_username}")

try:
    # Fetch user profile data
    user_response = requests.get(github_api_url_user)
    # To raise an HTTPError for bad responses (4xx or 5xx)
    user_response.raise_for_status()
    github_user_data = user_response.json()

    relevant_github_info = {
        "login": github_user_data.get("login"),
        "html_url": github_user_data.get("html_url"),
        "name": github_user_data.get("name"),
        "company": github_user_data.get("company"),
        "blog": github_user_data.get("blog"),
        "location": github_user_data.get("location"),
        "email": github_user_data.get("email"),
        "bio": github_user_data.get("bio"),
        "public_repos": github_user_data.get("public_repos"),
        "followers": github_user_data.get("followers"),
        "following": github_user_data.get("following"),
        "created_at": github_user_data.get("created_at"),
        "updated_at": github_user_data.get("updated_at")
    }
    print("GitHub user profile data fetched successfully.")

    # Fetch repo data
    repos_response = requests.get(github_api_url_repos)
    # To raise an HTTPError for bad responses (4xx or 5xx)
    repos_response.raise_for_status()
    github_repos_data = repos_response.json()
    print(f"Found {len(github_repos_data)} repositories.")

    repos_list = []
    languages_set = set()
    total_commits_all_repos = 0
    total_contributors_all_repos = 0
    num_repos_with_contributors_data = 0

    for repo in github_repos_data:
        repo_name = repo.get("name")
        repo_owner = repo.get("owner", {}).get("login")

        repo_info = {
            "name": repo_name,
            "html_url": repo.get("html_url"),
            "description": repo.get("description"),
            "language": repo.get("language"),
            "stargazers_count": repo.get("stargazers_count"),
            "forks_count": repo.get("forks_count"),
            "created_at": repo.get("created_at"),
            "updated_at": repo.get("updated_at"),
            "readme_content": None,
            "commit_activity_summary": {},
            "contributors_count": 0
        }

        # Fetch README content for each repo
        readme_url = f"https://api.github.com/repos/{repo_owner}/{repo_name}/readme"
        try:
            time.sleep(0.1)
            readme_response = requests.get(readme_url)
            if readme_response.status_code == 200:
                readme_data = readme_response.json()
                if 'content' in readme_data:
                    # Decode content which is base64 encoded
                    repo_info["readme_content"] = base64.b64decode(readme_data['content']).decode('utf-8')
            elif readme_response.status_code == 404:
                print(f"  No README found for {repo_name}.")
            else:
                print(f"  Error fetching README for {repo_name}: {readme_response.status_code}")
        except requests.exceptions.RequestException as e:
            print(f"  Network error fetching README for {repo_name}: {e}")

        # Fetch commit activity summary for each repo ---
        commit_activity_url = f"https://api.github.com/repos/{repo_owner}/{repo_name}/stats/commit_activity"
        try:
            time.sleep(0.1)
            commit_activity_response = requests.get(commit_activity_url)
            if commit_activity_response.status_code == 200:
                activity_data = commit_activity_response.json()
                repo_total_commits = sum(week['total'] for week in activity_data if 'total' in week)
                repo_info["commit_activity_summary"] = {"total_commits_past_year": repo_total_commits}
                total_commits_all_repos += repo_total_commits
            elif commit_activity_response.status_code == 202: # Data is being calculated
                print(f"  Commit activity for {repo_name} is still being calculated.")
            else:
                print(f"  Error fetching commit activity for {repo_name}: {commit_activity_response.status_code}")
        except requests.exceptions.RequestException as e:
            print(f"  Network error fetching commit activity for {repo_name}: {e}")

        # Fetch contributor count for each repository
        contributors_url = f"https://api.github.com/repos/{repo_owner}/{repo_name}/contributors"
        try:
            time.sleep(0.1)
            contributors_response = requests.get(contributors_url)
            if contributors_response.status_code == 200:
                contributors_data = contributors_response.json()
                repo_info["contributors_count"] = len(contributors_data)
                total_contributors_all_repos += len(contributors_data)
                num_repos_with_contributors_data += 1
            else:
                print(f"  Error fetching contributors for {repo_name}: {contributors_response.status_code}")
        except requests.exceptions.RequestException as e:
            print(f"  Network error fetching contributors for {repo_name}: {e}")

        repos_list.append(repo_info)
        if repo.get("language"):
            languages_set.add(repo.get("language"))

    relevant_github_info["repositories"] = repos_list
    relevant_github_info["languages_used"] = list(languages_set)
    relevant_github_info["total_commits_across_repos"] = total_commits_all_repos
    relevant_github_info["average_contributors_per_repo"] = (total_contributors_all_repos / num_repos_with_contributors_data) if num_repos_with_contributors_data > 0 else 0
    relevant_github_info["solo_coded_repos_count"] = sum(1 for repo in repos_list if repo.get("contributors_count") == 1)

    print("GitHub repositories, languages, READMEs, commit activity, and collaboration data fetched successfully.")

    json_file_path = '/content/drive/MyDrive/CMU/class (for Gemini use)/Resume (File responses)/rion_kurihara_resume.json'

    try:
        with open(json_file_path, 'r') as f:
            existing_data = json.load(f)
        print(f"Existing JSON data loaded from {json_file_path}")

        # Add GitHub data to the existing JSON
        existing_data["github_profile"] = relevant_github_info

        # Save the updated JSON data back to the file
        with open(json_file_path, 'w') as f:
            json.dump(existing_data, f, indent=4)
        print(f"GitHub data successfully added and saved to {json_file_path}")

    except FileNotFoundError:
        print(f"Error: JSON file not found at {json_file_path}. Please ensure it was created previously.")
    except json.JSONDecodeError:
        print(f"Error: Could not decode JSON from {json_file_path}. File might be corrupted or empty.")
    except Exception as e:
        print(f"Error processing or saving JSON file: {e}")

except requests.exceptions.HTTPError as errh:
    print(f"HTTP Error: {errh}")
except requests.exceptions.ConnectionError as errc:
    print(f"Error Connecting: {errc}")
except requests.exceptions.Timeout as errt:
    print(f"Timeout Error: {errt}")
except requests.exceptions.RequestException as err:
    print(f"An unexpected error occurred: {err}")

Fetching GitHub data for user: rionkuri22
GitHub user profile data fetched successfully.
Found 12 repositories.
  No README found for 15113-hw2-crossy-road.
  Commit activity for 15113-hw2-crossy-road is still being calculated.
  Commit activity for 15113-hw3-explore-api is still being calculated.
  Commit activity for 15113-hw4-sleep is still being calculated.
  Commit activity for 15113-hw6-sqlite is still being calculated.
  Commit activity for 15113-hw7-phase1 is still being calculated.
  Commit activity for 15113-hw7-phase2 is still being calculated.
  Commit activity for 15113-hw8-agentic is still being calculated.
  Commit activity for museo is still being calculated.
  Commit activity for project2_nfc is still being calculated.
  Commit activity for Re-collect_Scottylabs is still being calculated.
  Commit activity for recollect_website is still being calculated.
  Commit activity for rionkuri22.github.io is still being calculated.
GitHub repositories, languages, READMEs, commi